# 12 | Training-Only Class-Weighted Temporal CNN

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Objective:** Compare the same 1D-CNN with standard versus inverse-square-root class-weighted Cross-Entropy, using identical HARTH windows, participant split, random seed, architecture and training duration. Only the loss weighting changes. **This is an exploratory development ablation, not an independent Test or a matched Stage 9 benchmark.**

## 01. Import the established study functions

Training utilities remain in reusable `.py` modules, while this Notebook documents the experiment.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# Locate repository and register existing study modules.
root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root == root.parent: raise RuntimeError("Open from the project directory.")
    root = root.parent
sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))
from harth_data import HARTH_SUBJECTS, ACTIVITY_NAMES, prepare_harth_window_cohort
from core.data_splitting import split_group_indices
from core.temporal_cnn import train_temporal_cnn, compute_training_class_weights
from core.ml_baselines import classification_class_report, evaluate_predictions_by_subject
from core.group_cross_validation import compare_models_by_subject
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 02. Rebuild the exact bounded Stage 10/11 development cohort

Use all 29 development participants, with up to 500 label-blind uniformly sampled eligible windows per person. Original Test IDs are explicitly excluded. Source labels outside 1–8 are removed after valid windows are formed.

In [ ]:
# Keep identical source identities and window parameters across both runs.
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT_SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
TASK_CLASSES = tuple(range(1, 9))
X, y, metadata, overview = prepare_harth_window_cohort(
    DEVELOPMENT_SUBJECTS, root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES, max_windows_per_subject=500,
    window_size=100, step_size=50, sampling_rate=50,
)
assert len(DEVELOPMENT_SUBJECTS) == 29
assert not (set(metadata["subject_id"]) & RESERVED_TEST)
print("Shared raw IMU windows:", X.shape)
display(overview[["subject_id", "eligible_windows", "sampled_windows"]])

## 03. Fix the same participant-independent split

Train and Validation participants are identical to Stages 10 and 11. The extra development partition remains unused.

In [ ]:
# Split subjects, never overlap windows from a participant across partitions.
parts = split_group_indices(
    metadata["subject_id"].to_numpy(),
    train_fraction=0.70, validation_fraction=0.20,
    test_fraction=0.10, random_state=42,
)
train_idx, val_idx = parts["train"], parts["validation"]
X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]
val_subject_ids = metadata.iloc[val_idx]["subject_id"].to_numpy()
assert not (set(metadata.iloc[train_idx]["subject_id"]) & set(val_subject_ids))
print("Train windows:", len(y_train), "Validation windows:", len(y_val))
print("Unused development windows:", len(parts["test"]))

## 04. Inspect training-class imbalance and the proposed loss weights

Weights are computed **only from Train class counts** using inverse square roots, then normalised to mean one. Validation frequencies do not influence weights. These are heuristic loss weights, not patient-level prevalence estimates.

In [ ]:
# Display training counts and the fixed proposed loss weights.
training_counts = pd.Series(y_train).value_counts().reindex(TASK_CLASSES, fill_value=0)
weights = compute_training_class_weights(y_train, TASK_CLASSES, method="inverse_sqrt")
weight_table = pd.DataFrame({
    "activity": [ACTIVITY_NAMES[c] for c in TASK_CLASSES],
    "train_windows": training_counts.to_numpy(),
    "inverse_sqrt_weight": weights,
})
assert (training_counts > 0).all()
display(weight_table.round(4))

## 05. Train two identical CNN configurations

Both networks start from the same random seed and follow identical DataLoader shuffling and optimiser settings. Only the Cross-Entropy weight vector changes. Training is five epochs as in Stage 10.

In [ ]:
# Run unweighted and weighted models without any reserved Test observations.
settings = dict(
    X_train=X_train, y_train=y_train,
    X_validation=X_val, y_validation=y_val,
    classes=TASK_CLASSES, epochs=5, batch_size=128,
    learning_rate=1e-3, seed=42,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
runs = {}
for method, name in [
    ("none", "Standard CNN"),
    ("inverse_sqrt", "Weighted CNN"),
]:
    print("\nTraining:", name)
    model, history, predicted, scaler = train_temporal_cnn(
        **settings, class_weighting=method
    )
    runs[name] = {"model": model, "history": history, "predicted": predicted}
    print("Last-epoch Macro F1:", float(history.iloc[-1]["validation_macro_f1"]))
assert len(runs) == 2

## 06. Compare training curves and held-out Validation results

Macro F1 gives equal weight to the eight task classes. A weighted loss is not expected to improve every metric or every class.

In [ ]:
# Compare each last-epoch prediction on identical validation labels.
summary = pd.DataFrame([
    {"model": name,
     "macro_f1": f1_score(
         y_val, result["predicted"], labels=TASK_CLASSES,
         average="macro", zero_division=0),
     "balanced_accuracy": balanced_accuracy_score(y_val, result["predicted"]),
     "validation_windows": len(y_val)}
    for name, result in runs.items()
])
display(summary.round(4))

# Plot loss and validation Macro F1 with separate metric axes.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, result in runs.items():
    curve = result["history"]
    axes[0].plot(curve["epoch"], curve["train_loss"], marker="o", label=name)
    axes[1].plot(curve["epoch"], curve["validation_macro_f1"], marker="o", label=name)
axes[0].set(title="Training loss (different weighting objectives)",
            xlabel="Epoch", ylabel="Weighted or standard Cross-Entropy")
axes[1].set(title="Held-out Validation Macro F1", xlabel="Epoch",
            ylabel="Macro F1", ylim=(0, 1))
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## 07. Inspect minority-class performance and prediction errors

Report all eight classes and make missing predictions visible, particularly Shuffling and stair activities.

In [ ]:
# Show the exact precision, recall and F1 for both loss functions.
reports = []
for name, result in runs.items():
    report = classification_class_report(
        y_val, result["predicted"], classes=TASK_CLASSES,
        class_names=[ACTIVITY_NAMES[c] for c in TASK_CLASSES],
    )
    report["model"] = name
    reports.append(report)
class_table = pd.concat(reports, ignore_index=True)
display(class_table[["model", "class", "support", "precision", "recall", "f1"]].round(4))

# Plot model F1 bars side-by-side for each activity.
fig, ax = plt.subplots(figsize=(12, 5))
positions = np.arange(len(TASK_CLASSES))
for i, name in enumerate(runs):
    values = class_table.loc[class_table["model"] == name, "f1"].to_numpy()
    ax.bar(positions + (i - 0.5) * 0.36, values, width=0.34, label=name)
ax.set_xticks(positions, [ACTIVITY_NAMES[c] for c in TASK_CLASSES], rotation=40, ha="right")
ax.set(title="Per-class F1: standard vs training-weighted CNN",
       ylabel="F1 score", ylim=(0, 1))
ax.grid(axis="y", alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()

# Display both confusion matrices on the same true/predicted label axes.
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, (name, result) in zip(axes, runs.items()):
    matrix = confusion_matrix(y_val, result["predicted"], labels=TASK_CLASSES)
    ConfusionMatrixDisplay(
        confusion_matrix=matrix, display_labels=[ACTIVITY_NAMES[c] for c in TASK_CLASSES],
    ).plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
    ax.set_title(name)
fig.tight_layout()
plt.show()

## 08. Check paired participant-level changes

Every individual contributes one equal-weight comparison score; overlapping windows are not treated as independent study participants. This comparison is descriptive, not a statistical significance test.

In [ ]:
# Compute fixed-eight-class Macro F1 for the same held-out participants.
individual_results = pd.concat([
    evaluate_predictions_by_subject(
        y_val, result["predicted"], val_subject_ids,
        TASK_CLASSES, model_name=name,
    )
    for name, result in runs.items()
], ignore_index=True)
paired, paired_summary = compare_models_by_subject(
    individual_results, model_a="Weighted CNN", model_b="Standard CNN",
    score_column="macro_f1_fixed_task",
)
display(paired.round(4))
display(pd.DataFrame([paired_summary]).round(4))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(paired["subject_id"], paired["difference_a_minus_b"])
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Participant-level Macro F1: weighted minus standard",
       xlabel="Held-out participant", ylabel="Signed Macro F1 difference")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()

## 09. Research quality gate

This is a controlled training-loss ablation on a previously inspected development Validation split. Findings are exploratory and should be followed by a broader group-CV design. Do not claim the held-out original Test has been independently evaluated.

In [ ]:
# Validate class specifications, identical predictions and subject boundaries.
assert len(X_train) == len(y_train) and len(X_val) == len(y_val)
assert set(np.unique(y_train)) == set(TASK_CLASSES)
assert not (set(val_subject_ids) & RESERVED_TEST)
assert all(len(item["predicted"]) == len(y_val) for item in runs.values())
assert len(paired) == len(set(val_subject_ids))
assert summary["macro_f1"].between(0, 1).all()
print("Training-loss ablation quality gate: PASSED")
print("Reserved original Test loaded/scored: NO")
print("Independent validation of weight benefits: NOT ESTABLISHED")